# 01 · Target Factory — the targets, the panel and the model input

**What this stage is for.** It builds the target of every unit-period (`time.period`: month,
week or day) from the target source by the rule in `data.target` (or takes an upstream column),
joins the sources into one panel and saves the **model input**: the sources joined plus one
target column per use case, `target_<use_case>`, the one table every later stage reads
(`docs/components/dataset.md`, D-131). Everything else the config derives from it (eligibility,
the candidates, the contract checks, the client split) is computed on read from the reader's
config, so those may change without a rebuild; one table may serve several use cases that share
the sources, keys and period (`run a.yml b.yml --stages target_factory`). Saving it freezes the
config's data part (sources, keys, period, target); later changes go through the decision audit
log (`record_decision`, D-125). To work on another use case, change the config path in the
setup cell and rerun.

**What to look at.**
1. **The target summary** (step 1): rows per period; `null_share`, the targets unknown at the
   panel edge, for leavers and across gaps; `null_value_share`, the share of complete windows
   that held a null in the target column (a null never counts as an event: a null that means a
   value is filled upstream); and, for the observation window only, the mean target, the
   positive rows and the positive events. A rate that drifts across the window, or a null share
   above a few percent, is worth a question upstream before anything is modelled.
2. **The join printout** (step 2): the share of the spine's rows each source matched per period
   and the null share per target. A source that matches few rows usually means a key or a
   period tag that differs between sources.
3. **The window frame** (step 3): what EDA will see, read back from the model input with the
   eligibility, contract and split lines printing as they are derived: every client after the
   analysis filters with the modelling tags as columns, and the collect printout against the
   budget. A filter that drops nothing, or almost everything, usually means a wrong condition.

**Decisions to record:** none here. The data definition is the config itself: if the targets
are wrong, change the config and rebuild; if the eligibility or the candidates are wrong, change
the config and rerun from EDA, the table stays. The data-definition hash changes either way.

**Docs:** `docs/components/target_factory.md` · `docs/components/dataset.md` ·
`docs/components/config.md` · the next stage: `02_eda` (M2.2).


In [ ]:
from pathlib import Path

import core_modelling
from core_modelling.config import load_config
from core_modelling.dataset import build_frames, save_model_input
from core_modelling.environment import get_spark_session, load_environment
from core_modelling.target_factory import run_target_factory

# The use case and where the framework runs: absolute paths, never relative to the working
# directory (T5). The repo's example is found from the editable install; a use-case repo writes
# its own absolute config path here and reads the environment from $CORE_MODELLING_ENVIRONMENT.
REPO = Path(core_modelling.__file__).resolve().parents[2]
CONFIG = REPO / "conf/base/cc_acquisition.yml"
ENTITY = "entity_a"
ENVIRONMENT = REPO / "conf/local/environment.yml"  # or None: $CORE_MODELLING_ENVIRONMENT

env = load_environment(str(ENVIRONMENT) if ENVIRONMENT else None)
spark = get_spark_session(env)
cfg = load_config(CONFIG)
print(cfg.use_case, cfg.task, cfg.data.target, cfg.time, sep="\n")

## 1. The targets
The summary covers every row of the target source, before eligibility. Check the per-period rows
and null shares first, then the window periods' rate: the `all` row pools the window.


In [ ]:
summary = run_target_factory(cfg, ENTITY, env, spark)

## 2. Save the model input
The sources are joined (the join lines print as it goes) and written with this use case's target
column to the path the environment file maps for `data.model_input`, with `_model_input.json` in
the same folder (the shared sources, keys and period, the periods covered, the join log and, per
use case, the target-spec hash, the lineage, the window and the null share). `save_model_input`
takes one config or a list of configs that share the sources, keys and period. From here on the
config's data part is frozen.


In [ ]:
path = save_model_input(cfg, ENTITY, env, spark)

## 3. The window frame
What EDA will see, read back from the model input: the eligibility, contract and split lines
print as they are derived from this config. Every client after the analysis filters, with
`unit_id`, `period`, the candidates, the modelling tags (here `non_holders`), `eligible`,
`target` and `split` as columns. The share tagged per period is the eligibility funnel; the
collect printout says how big the frame is against the budget.


In [ ]:
frames = build_frames(cfg, ENTITY, env, spark, stage="eda")
window = frames.window
print(
    len(window),
    "rows,",
    len(frames.meta["candidates"]),
    "candidate columns;",
    "lineage excluded:",
    frames.meta["lineage"],
)
window.groupby("period")[[*frames.meta["tags"], "eligible"]].mean()

In [ ]:
window.head()